In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path



In [2]:
# Define paths
DATA_PATH = Path("../../../../data/raw")

In [3]:
# Load data
df = pd.read_excel(DATA_PATH / "2020_Birth_Final.xlsx")

In [4]:
# ============================================
# INSTALL REQUIRED PACKAGES (if not already installed)
# ============================================
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement
except ImportError:
    print("Installing python-docx...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'python-docx'])
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================
# SET CORRECT COLUMN NAMES
# ============================================

# CHANGED: Birth Order column name (adjust based on your actual column name)
BIRTH_ORDER_COL = 'Birth_Order'  # Change this to your actual column name
MOTHER_RACE_COL = 'Race_of_Mother'
FATHER_RACE_COL = 'Race_of_Father'
DISTRICT_COL = 'Registered_District'

print("=" * 80)
print("🏥 MISSING BIRTH ORDER ANALYSIS BY DISTRICT AND ETHNICITY")
print("=" * 80)

# ============================================
# STEP 1: IDENTIFY MISSING BIRTH ORDER COUNTS
# ============================================

print("\n📊 STEP 1: Missing Birth Order Analysis")
print("-" * 80)

# Check if column exists
if BIRTH_ORDER_COL not in df.columns:
    print(f"❌ ERROR: Column '{BIRTH_ORDER_COL}' not found!")
    print(f"Available columns: {list(df.columns)}")
    print(f"\n💡 TIP: Please update BIRTH_ORDER_COL with the correct column name")
else:
    # Create missing birth order indicator
    df['Missing_Birth_Order'] = df[BIRTH_ORDER_COL].isna()
    
    # Overall missing statistics
    total_records = len(df)
    total_missing = df['Missing_Birth_Order'].sum()
    total_complete = total_records - total_missing
    
    print(f"\nOverall Statistics:")
    print(f"   • Total records: {total_records:,}")
    print(f"   • Missing birth order: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    print(f"   • Complete birth order: {total_complete:,} ({total_complete/total_records*100:.2f}%)")

# ============================================
# STEP 2: ETHNICITY STANDARDIZATION (IUPAC STANDARDS)
# ============================================

print("\n📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)")
print("-" * 80)

# Define ethnicities with IUPAC codes following international standards
ETHNICITIES = {
    'Sinhalese': {'code': 'SIN', 'full_name': 'Sinhalese', 'iupac_name': 'Sinhala'},
    'Srilankan Tamil': {'code': 'TAM_SL', 'full_name': 'Sri Lankan Tamil', 'iupac_name': 'Tamil (Sri Lanka)'},
    'Indian Tamil': {'code': 'TAM_IN', 'full_name': 'Indian Tamil', 'iupac_name': 'Tamil (India)'},
    'Srilankan Moor': {'code': 'MOOR_SL', 'full_name': 'Sri Lankan Moor', 'iupac_name': 'Moor (Sri Lanka)'},
    'Burgher': {'code': 'BUR', 'full_name': 'Burgher', 'iupac_name': 'Burgher'},
    'Malay': {'code': 'MAL', 'full_name': 'Malay', 'iupac_name': 'Malay'},
    'Srilankan Chetty': {'code': 'CHT_SL', 'full_name': 'Sri Lankan Chetty', 'iupac_name': 'Chetty'},
    'Bharatha': {'code': 'BHA', 'full_name': 'Bharatha', 'iupac_name': 'Bharatha'},
    'Indian Moor': {'code': 'MOOR_IN', 'full_name': 'Indian Moor', 'iupac_name': 'Moor (India)'},
    'Pakistan Moor': {'code': 'MOOR_PK', 'full_name': 'Pakistan Moor', 'iupac_name': 'Moor (Pakistan)'},
    'Other Foreigners': {'code': 'OTH_FGN', 'full_name': 'Other Foreigners', 'iupac_name': 'Other Nationalities'},
    'Other Srilankans': {'code': 'OTH_SL', 'full_name': 'Other Sri Lankans', 'iupac_name': 'Other Ethnic Groups'}
}

# Standardize ethnicity function
def standardize_ethnicity(race):
    """Standardize ethnicity names following IUPAC nomenclature"""
    if pd.isna(race):
        return None
    
    # Convert to string and strip
    race_str = str(race).strip()
    
    # Mapping for numeric codes (1-13) as per Sri Lanka vital statistics
    code_map = {
        '1': 'Sinhalese',
        '2': 'Srilankan Tamil',
        '3': 'Indian Tamil',
        '4': 'Srilankan Moor',
        '5': 'Burgher',
        '6': 'Malay',
        '7': 'Srilankan Chetty',
        '8': 'Bharatha',
        '9': 'Indian Moor',
        '10': 'Pakistan Moor',
        '11': 'Other Foreigners',
        '12': 'Other Srilankans'
    }
    
    # Mapping for text values
    text_map = {
        'Sinhalese': 'Sinhalese',
        'Srilankan Tamil': 'Srilankan Tamil',
        'Sri Lankan Tamil': 'Srilankan Tamil',
        'Indian Tamil': 'Indian Tamil',
        'Srilankan Moor': 'Srilankan Moor',
        'Sri Lankan Moor': 'Srilankan Moor',
        'Moor': 'Srilankan Moor',
        'Burgher': 'Burgher',
        'Malay': 'Malay',
        'Srilankan Chetty': 'Srilankan Chetty',
        'Bharatha': 'Bharatha',
        'Indian Moor': 'Indian Moor',
        'Pakistan Moor': 'Pakistan Moor',
        'Other Foreigners': 'Other Foreigners',
        'Other Srilankans': 'Other Srilankans'
    }
    
    # Check if it's a numeric code
    if race_str in code_map:
        return code_map[race_str]
    
    # Check if it's a text value
    return text_map.get(race_str, None)

# Create standardized ethnicity columns
df['Mother_Ethnicity_Std'] = df[MOTHER_RACE_COL].apply(standardize_ethnicity)
df['Father_Ethnicity_Std'] = df[FATHER_RACE_COL].apply(standardize_ethnicity)

# Show unique ethnicities found
unique_ethnicities = df['Mother_Ethnicity_Std'].dropna().unique()
print(f"\nUnique ethnicities found in dataset: {sorted(unique_ethnicities)}")

# ============================================
# STEP 3: DISTRICT-LEVEL MISSING BIRTH ORDER ANALYSIS
# ============================================

print("\n📊 STEP 3: District-Level Missing Birth Order Analysis")
print("-" * 80)

# Get unique districts
districts = df[DISTRICT_COL].dropna().unique()
district_summary = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    
    total_births = len(district_data)
    # CHANGED: Now using Missing_Birth_Order
    missing_order = district_data['Missing_Birth_Order'].sum()
    complete_order = total_births - missing_order
    
    district_summary.append({
        'District': district,
        'Total_Births': total_births,
        'Missing_Order': missing_order,
        'Complete_Order': complete_order,
        'Missing_Rate': (missing_order / total_births * 100) if total_births > 0 else 0
    })

# Create district summary DataFrame
district_df = pd.DataFrame(district_summary)
district_df = district_df.sort_values('Missing_Rate', ascending=False)

# ============================================
# STEP 4: ETHNICITY DISTRIBUTION BY DISTRICT
# ============================================

print("\n📊 STEP 4: Ethnicity Distribution by District")
print("-" * 80)

# Create detailed ethnicity-district analysis
district_ethnicity_analysis = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    district_total = len(district_data)
    
    if district_total == 0:
        continue
    
    for ethnicity, config in ETHNICITIES.items():
        ethnic_mothers = district_data[district_data['Mother_Ethnicity_Std'] == ethnicity].shape[0]
        # CHANGED: Now using Missing_Birth_Order
        ethnic_missing = district_data[
            (district_data['Mother_Ethnicity_Std'] == ethnicity) & 
            (district_data['Missing_Birth_Order'] == True)
        ].shape[0]
        
        if ethnic_mothers > 0:
            pct_of_district = (ethnic_mothers / district_total * 100)
            pct_of_district_missing = (ethnic_missing / district_total * 100)
            missing_rate_in_ethnic = (ethnic_missing / ethnic_mothers * 100)
            
            district_ethnicity_analysis.append({
                'District': district,
                'Ethnicity': ethnicity,
                'IUPAC_Code': config['code'],
                'IUPAC_Name': config['iupac_name'],
                'Total_Mothers': ethnic_mothers,
                'Pct_of_District_Total': round(pct_of_district, 2),
                'Missing_Order': ethnic_missing,  # CHANGED
                'Missing_Rate_in_Ethnic': round(missing_rate_in_ethnic, 2),
                'Pct_of_District_Missing': round(pct_of_district_missing, 2)
            })

# Create DataFrame
if district_ethnicity_analysis:
    ethnicity_district_df = pd.DataFrame(district_ethnicity_analysis)
    
    # ============================================
    # CREATE WORD DOCUMENT WITH IUPAC STANDARDS
    # ============================================
    
    print("\n📄 Creating Word Document with IUPAC Standards...")
    
    # Create new document
    doc = Document()
    
    # Set document margins
    sections = doc.sections
    for section in sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)
    
    # Add title - CHANGED
    title = doc.add_heading('Missing Birth Order Analysis Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add subtitle with IUPAC standards
    subtitle = doc.add_heading('Sri Lanka Vital Statistics - IUPAC Compliant Nomenclature', 2)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add date
    date_para = doc.add_paragraph(f'Report Generated: {datetime.now().strftime("%B %d, %Y")}')
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 1: EXECUTIVE SUMMARY
    # ============================================
    
    doc.add_heading('1. Executive Summary', level=1)
    
    # CHANGED: Updated summary text
    summary_text = f"""
    This report presents a comprehensive analysis of missing birth order data in Sri Lanka, 
    stratified by district and ethnicity. The analysis follows IUPAC (International Union of 
    Pure and Applied Chemistry) standards for ethnic nomenclature to ensure international 
    consistency and scientific rigor.
    
    Key Findings:
    • Total Records Analyzed: {total_records:,}
    • Missing Birth Order Records: {total_missing:,} ({total_missing/total_records*100:.2f}%)
    • Complete Records: {total_complete:,} ({total_complete/total_records*100:.2f}%)
    • Number of Districts Analyzed: {len(districts)}
    • Number of Ethnic Groups Identified: {len(unique_ethnicities)}
    
    Note: Birth order refers to the order of birth (first child, second child, etc.) and is a 
    critical demographic variable for understanding fertility patterns and family planning needs.
    """
    
    doc.add_paragraph(summary_text)
    
    # ============================================
    # SECTION 2: IUPAC ETHNICITY CLASSIFICATION
    # ============================================
    
    doc.add_heading('2. IUPAC Ethnicity Classification System', level=1)
    doc.add_paragraph('The following ethnicity codes follow IUPAC standards for population genetics and vital statistics reporting:')
    
    # Create ethnicity table
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Light Grid Accent 1'
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'IUPAC Standard Name'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
        row_cells[2].text = config['iupac_name']
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 3: DISTRICT-WISE ANALYSIS
    # ============================================
    
    doc.add_heading('3. District-Wise Missing Birth Order Analysis', level=1)
    
    # Add district summary table - CHANGED
    doc.add_heading('3.1 District Summary Statistics', level=2)
    
    district_table = doc.add_table(rows=1, cols=4)
    district_table.style = 'Light Grid Accent 1'
    hdr_cells = district_table.rows[0].cells
    hdr_cells[0].text = 'District'
    hdr_cells[1].text = 'Total Births'
    hdr_cells[2].text = 'Missing Order Records'
    hdr_cells[3].text = 'Missing Rate (%)'
    
    for _, row in district_df.iterrows():
        row_cells = district_table.add_row().cells
        row_cells[0].text = row['District']
        row_cells[1].text = f"{row['Total_Births']:,}"
        row_cells[2].text = f"{row['Missing_Order']:,}"
        row_cells[3].text = f"{row['Missing_Rate']:.2f}%"
    
    # Add top districts with highest missing rates
    doc.add_heading('3.2 Districts with Highest Missing Rates', level=2)
    
    top_districts = district_df.head(10)
    top_table = doc.add_table(rows=1, cols=3)
    top_table.style = 'Light Grid Accent 1'
    hdr_cells = top_table.rows[0].cells
    hdr_cells[0].text = 'Rank'
    hdr_cells[1].text = 'District'
    hdr_cells[2].text = 'Missing Rate (%)'
    
    for idx, (_, row) in enumerate(top_districts.iterrows(), 1):
        row_cells = top_table.add_row().cells
        row_cells[0].text = str(idx)
        row_cells[1].text = row['District']
        row_cells[2].text = f"{row['Missing_Rate']:.2f}%"
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 4: DETAILED DISTRICT-ETHNICITY ANALYSIS
    # ============================================
    
    doc.add_heading('4. Detailed District-Ethnicity Analysis', level=1)
    doc.add_paragraph('The following tables show the ethnicity distribution and missing birth order patterns for each district, following IUPAC nomenclature.')
    
    for district in districts[:15]:  # Show first 15 districts (adjust as needed)
        district_ethnic = ethnicity_district_df[ethnicity_district_df['District'] == district]
        
        if len(district_ethnic) > 0:
            district_total = district_df[district_df['District'] == district]['Total_Births'].values[0]
            district_missing = district_df[district_df['District'] == district]['Missing_Order'].values[0]
            
            # Add district header
            doc.add_heading(f'{district}', level=2)
            doc.add_paragraph(f'Total Births: {district_total:,} | Missing Order Records: {district_missing:,} ({district_missing/district_total*100:.2f}%)')
            
            # Create ethnicity table for district - CHANGED columns
            ethnic_table = doc.add_table(rows=1, cols=7)
            ethnic_table.style = 'Light Grid Accent 1'
            hdr_cells = ethnic_table.rows[0].cells
            hdr_cells[0].text = 'IUPAC Code'
            hdr_cells[1].text = 'Ethnicity'
            hdr_cells[2].text = 'Count'
            hdr_cells[3].text = '% of District'
            hdr_cells[4].text = 'Missing Order'
            hdr_cells[5].text = 'Missing Rate (%)'
            hdr_cells[6].text = '% of District Missing'
            
            district_ethnic_sorted = district_ethnic.sort_values('Pct_of_District_Total', ascending=False)
            
            for _, row in district_ethnic_sorted.iterrows():
                row_cells = ethnic_table.add_row().cells
                row_cells[0].text = row['IUPAC_Code']
                row_cells[1].text = row['Ethnicity']
                row_cells[2].text = f"{row['Total_Mothers']:,}"
                row_cells[3].text = f"{row['Pct_of_District_Total']:.2f}%"
                row_cells[4].text = f"{row['Missing_Order']:,}"  # CHANGED
                row_cells[5].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[6].text = f"{row['Pct_of_District_Missing']:.2f}%"
            
            # Add district summary
            doc.add_paragraph()
            most_prevalent = district_ethnic_sorted.iloc[0]
            highest_missing = district_ethnic_sorted.loc[district_ethnic_sorted['Missing_Rate_in_Ethnic'].idxmax()]
            largest_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_District_Missing'].idxmax()]
            
            summary_para = doc.add_paragraph()
            summary_para.add_run('District Summary:').bold = True
            doc.add_paragraph(f'• Most prevalent ethnicity: {most_prevalent["Ethnicity"]} ({most_prevalent["IUPAC_Code"]}) - {most_prevalent["Pct_of_District_Total"]:.1f}% of district')
            doc.add_paragraph(f'• Highest missing rate: {highest_missing["Ethnicity"]} ({highest_missing["IUPAC_Code"]}) - {highest_missing["Missing_Rate_in_Ethnic"]:.1f}% missing')
            doc.add_paragraph(f'• Largest contributor to missing data: {largest_contributor["Ethnicity"]} ({largest_contributor["IUPAC_Code"]}) - {largest_contributor["Pct_of_District_Missing"]:.1f}% of missing records')
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 5: ETHNICITY-SPECIFIC ANALYSIS
    # ============================================
    
    doc.add_heading('5. Ethnicity-Specific Analysis', level=1)
    
    # Calculate overall ethnicity statistics - CHANGED
    ethnicity_overall = ethnicity_district_df.groupby(['Ethnicity', 'IUPAC_Code', 'IUPAC_Name']).agg({
        'Total_Mothers': 'sum',
        'Missing_Order': 'sum'  # CHANGED
    }).reset_index()
    ethnicity_overall['Missing_Rate'] = (ethnicity_overall['Missing_Order'] / ethnicity_overall['Total_Mothers'] * 100)
    ethnicity_overall = ethnicity_overall.sort_values('Missing_Rate', ascending=False)
    
    doc.add_heading('5.1 Overall Ethnicity Statistics', level=2)
    
    overall_table = doc.add_table(rows=1, cols=5)
    overall_table.style = 'Light Grid Accent 1'
    hdr_cells = overall_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'Total Births'
    hdr_cells[3].text = 'Missing Order Records'
    hdr_cells[4].text = 'Missing Rate (%)'
    
    for _, row in ethnicity_overall.iterrows():
        row_cells = overall_table.add_row().cells
        row_cells[0].text = row['IUPAC_Code']
        row_cells[1].text = row['Ethnicity']
        row_cells[2].text = f"{row['Total_Mothers']:,}"
        row_cells[3].text = f"{row['Missing_Order']:,}"  # CHANGED
        row_cells[4].text = f"{row['Missing_Rate']:.2f}%"
    
    # Add ethnicity-specific district analysis
    doc.add_heading('5.2 Ethnicity-Specific District Analysis', level=2)
    
    for ethnicity in list(ETHNICITIES.keys())[:6]:  # Show first 6 ethnicities
        ethnic_data = ethnicity_district_df[ethnicity_district_df['Ethnicity'] == ethnicity]
        
        if len(ethnic_data) > 0:
            ethnic_total = ethnic_data['Total_Mothers'].sum()
            ethnic_missing = ethnic_data['Missing_Order'].sum()  # CHANGED
            ethnic_rate = (ethnic_missing / ethnic_total * 100) if ethnic_total > 0 else 0
            
            doc.add_heading(f'{ETHNICITIES[ethnicity]["full_name"]} ({ETHNICITIES[ethnicity]["code"]})', level=3)
            doc.add_paragraph(f'Overall Statistics: Total Births: {ethnic_total:,} | Missing Order: {ethnic_missing:,} ({ethnic_rate:.2f}%)')
            
            # Create table for districts with highest missing rates for this ethnicity
            ethnic_table = doc.add_table(rows=1, cols=4)
            ethnic_table.style = 'Light Grid Accent 1'
            hdr_cells = ethnic_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Order Records'
            hdr_cells[3].text = 'Missing Rate (%)'
            
            ethnic_sorted = ethnic_data.sort_values('Missing_Rate_in_Ethnic', ascending=False).head(10)
            
            for _, row in ethnic_sorted.iterrows():
                row_cells = ethnic_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Order']:,}"  # CHANGED
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 6: CONCLUSIONS AND RECOMMENDATIONS
    # ============================================
    
    doc.add_heading('6. Conclusions and Recommendations', level=1)
    
    # Find key insights
    worst_district = district_df.iloc[0]
    worst_ethnicity = ethnicity_overall.iloc[0]
    
    # CHANGED: Updated conclusions
    conclusions = f"""
    6.1 Key Findings
    
    • Data Completeness: {total_complete/total_records*100:.2f}% of birth records have complete birth order data,
      indicating {total_missing/total_records*100:.2f}% of records require data quality improvement.
    
    • Geographic Disparities: {worst_district['District']} shows the highest missing rate for birth order at 
      {worst_district['Missing_Rate']:.2f}%, suggesting potential data collection challenges in this region.
    
    • Ethnic Disparities: {worst_ethnicity['Ethnicity']} ({worst_ethnicity['IUPAC_Code']}) has the highest 
      missing rate for birth order at {worst_ethnicity['Missing_Rate']:.2f}%, indicating potential systematic bias 
      in data collection across ethnic groups.
    
    • Public Health Implications: Missing birth order data affects the accuracy of fertility analyses, 
      family planning assessments, and maternal and child health program evaluations.
    
    6.2 Recommendations
    
    1. Standardize Data Collection Protocols: Implement uniform birth order recording procedures across all districts,
       particularly in high-missing-rate regions.
    
    2. Ethnicity-Specific Interventions: Develop targeted data quality improvement programs for ethnic groups 
       with high missing rates, respecting cultural and linguistic sensitivities.
    
    3. IUPAC Compliance: Maintain adherence to IUPAC ethnic nomenclature standards to ensure international 
       comparability and scientific rigor.
    
    4. Regular Monitoring: Establish quarterly data quality monitoring systems to track improvements in 
       missing birth order rates by district and ethnicity.
    
    5. Capacity Building: Provide training for healthcare workers on the importance of complete birth order 
       documentation, especially in districts with high missing rates.
    
    6. Electronic Health Records: Implement or strengthen electronic health record systems with mandatory 
       birth order fields to reduce missing data.
    """
    
    doc.add_paragraph(conclusions)
    
    # Add methodology section
    doc.add_heading('7. Methodology', level=1)
    
    methodology = f"""
    This analysis was conducted using vital statistics data from Sri Lanka. The methodology follows 
    IUPAC standards for ethnic classification and WHO guidelines for demographic data collection.
    
    Data Sources:
    • Birth Registration Records: {total_records:,} records
    • Time Period: Full dataset analysis
    • Geographic Coverage: {len(districts)} districts
    
    Birth Order Definition:
    Birth order refers to the numerical order of a child's birth among all live births to the same mother 
    (first child, second child, third child, etc.). Complete birth order data is essential for:
    • Fertility rate calculations
    • Family planning program evaluation
    • Maternal health risk assessment
    • Demographic transition analysis
    
    Ethnic Classification:
    Ethnicities were classified according to IUPAC standards using the following codes:
    """
    
    doc.add_paragraph(methodology)
    
    # Add IUPAC code reference
    ref_table = doc.add_table(rows=1, cols=2)
    ref_table.style = 'Light Grid Accent 1'
    hdr_cells = ref_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnic Group'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = ref_table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
    
    # Save the document - CHANGED filename
    filename = f'Missing_Birth_Order_Analysis_IUPAC_{datetime.now().strftime("%Y%m%d_%H%M%S")}.docx'
    doc.save(filename)
    print(f"\n✅ Word document saved as: {filename}")
    
    # ============================================
    # STEP 5: DISPLAY SUMMARY IN CONSOLE
    # ============================================
    
    print("\n" + "=" * 100)
    print("📊 FINAL SUMMARY: Missing Birth Order Analysis by District and Ethnicity")
    print("=" * 100)
    
    print(f"\n✅ Analysis Complete!")
    print(f"✅ Word Document Generated: {filename}")
    print(f"\nOverall Statistics:")
    print(f"   • Total Districts: {len(districts)}")
    print(f"   • Total Births: {total_records:,}")
    print(f"   • Total Missing Birth Order: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    
    print(f"\n🏆 Top 5 Districts with Highest Missing Rate:")
    for _, row in district_df.head(5).iterrows():
        print(f"   • {row['District']}: {row['Missing_Rate']:.2f}% ({row['Missing_Order']:,}/{row['Total_Births']:,})")
    
    print(f"\n🏆 Top 5 Ethnicities with Highest Missing Rate (Overall):")
    for _, row in ethnicity_overall.head(5).iterrows():
        print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Missing_Rate']:.2f}% ({row['Missing_Order']:,}/{row['Total_Mothers']:,})")

else:
    print("\n⚠️ No ethnicity data found. Please check the ethnicity column values.")
    print(f"Sample values from {MOTHER_RACE_COL}:")
    print(df[MOTHER_RACE_COL].value_counts().head(10))

print("\n✅ Analysis Complete!")

🏥 MISSING BIRTH ORDER ANALYSIS BY DISTRICT AND ETHNICITY

📊 STEP 1: Missing Birth Order Analysis
--------------------------------------------------------------------------------

Overall Statistics:
   • Total records: 301,691
   • Missing birth order: 27,213 (9.02%)
   • Complete birth order: 274,478 (90.98%)

📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)
--------------------------------------------------------------------------------

Unique ethnicities found in dataset: ['Burgher', 'Indian Tamil', 'Malay', 'Sinhalese', 'Srilankan Moor', 'Srilankan Tamil']

📊 STEP 3: District-Level Missing Birth Order Analysis
--------------------------------------------------------------------------------

📊 STEP 4: Ethnicity Distribution by District
--------------------------------------------------------------------------------

📄 Creating Word Document with IUPAC Standards...

✅ Word document saved as: Missing_Birth_Order_Analysis_IUPAC_20260501_201353.docx

📊 FINAL SUMMARY: Missing Birth

In [5]:
# ============================================
# INSTALL REQUIRED PACKAGES (if not already installed)
# ============================================
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement
except ImportError:
    print("Installing python-docx...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'python-docx'])
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================
# SET CORRECT COLUMN NAMES
# ============================================

# CHANGED: Birth Order column name (adjust based on your actual column name)
BIRTH_ORDER_COL = 'Birth_Order'  # Change this to your actual column name
MOTHER_RACE_COL = 'Race_of_Mother'
FATHER_RACE_COL = 'Race_of_Father'
DISTRICT_COL = 'Registered_District'

print("=" * 80)
print("🏥 MISSING BIRTH ORDER ANALYSIS BY DISTRICT AND ETHNICITY")
print("=" * 80)

# ============================================
# STEP 1: IDENTIFY MISSING BIRTH ORDER COUNTS
# ============================================

print("\n📊 STEP 1: Missing Birth Order Analysis")
print("-" * 80)

# Check if column exists
if BIRTH_ORDER_COL not in df.columns:
    print(f"❌ ERROR: Column '{BIRTH_ORDER_COL}' not found!")
    print(f"Available columns: {list(df.columns)}")
    print(f"\n💡 TIP: Please update BIRTH_ORDER_COL with the correct column name")
else:
    # Create missing birth order indicator
    df['Missing_Birth_Order'] = df[BIRTH_ORDER_COL].isna()
    
    # Overall missing statistics
    total_records = len(df)
    total_missing = df['Missing_Birth_Order'].sum()
    total_complete = total_records - total_missing
    
    print(f"\nOverall Statistics:")
    print(f"   • Total records: {total_records:,}")
    print(f"   • Missing birth order: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    print(f"   • Complete birth order: {total_complete:,} ({total_complete/total_records*100:.2f}%)")

# ============================================
# STEP 2: ETHNICITY STANDARDIZATION (IUPAC STANDARDS)
# ============================================

print("\n📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)")
print("-" * 80)

# Define ethnicities with IUPAC codes following international standards
ETHNICITIES = {
    'Sinhalese': {'code': 'SIN', 'full_name': 'Sinhalese', 'iupac_name': 'Sinhala'},
    'Srilankan Tamil': {'code': 'TAM_SL', 'full_name': 'Sri Lankan Tamil', 'iupac_name': 'Tamil (Sri Lanka)'},
    'Indian Tamil': {'code': 'TAM_IN', 'full_name': 'Indian Tamil', 'iupac_name': 'Tamil (India)'},
    'Srilankan Moor': {'code': 'MOOR_SL', 'full_name': 'Sri Lankan Moor', 'iupac_name': 'Moor (Sri Lanka)'},
    'Burgher': {'code': 'BUR', 'full_name': 'Burgher', 'iupac_name': 'Burgher'},
    'Malay': {'code': 'MAL', 'full_name': 'Malay', 'iupac_name': 'Malay'},
    'Srilankan Chetty': {'code': 'CHT_SL', 'full_name': 'Sri Lankan Chetty', 'iupac_name': 'Chetty'},
    'Bharatha': {'code': 'BHA', 'full_name': 'Bharatha', 'iupac_name': 'Bharatha'},
    'Indian Moor': {'code': 'MOOR_IN', 'full_name': 'Indian Moor', 'iupac_name': 'Moor (India)'},
    'Pakistan Moor': {'code': 'MOOR_PK', 'full_name': 'Pakistan Moor', 'iupac_name': 'Moor (Pakistan)'},
    'Other Foreigners': {'code': 'OTH_FGN', 'full_name': 'Other Foreigners', 'iupac_name': 'Other Nationalities'},
    'Other Srilankans': {'code': 'OTH_SL', 'full_name': 'Other Sri Lankans', 'iupac_name': 'Other Ethnic Groups'}
}

# Standardize ethnicity function
def standardize_ethnicity(race):
    """Standardize ethnicity names following IUPAC nomenclature"""
    if pd.isna(race):
        return None
    
    # Convert to string and strip
    race_str = str(race).strip()
    
    # Mapping for numeric codes (1-13) as per Sri Lanka vital statistics
    code_map = {
        '1': 'Sinhalese',
        '2': 'Srilankan Tamil',
        '3': 'Indian Tamil',
        '4': 'Srilankan Moor',
        '5': 'Burgher',
        '6': 'Malay',
        '7': 'Srilankan Chetty',
        '8': 'Bharatha',
        '9': 'Indian Moor',
        '10': 'Pakistan Moor',
        '11': 'Other Foreigners',
        '12': 'Other Srilankans'
    }
    
    # Mapping for text values
    text_map = {
        'Sinhalese': 'Sinhalese',
        'Srilankan Tamil': 'Srilankan Tamil',
        'Sri Lankan Tamil': 'Srilankan Tamil',
        'Indian Tamil': 'Indian Tamil',
        'Srilankan Moor': 'Srilankan Moor',
        'Sri Lankan Moor': 'Srilankan Moor',
        'Moor': 'Srilankan Moor',
        'Burgher': 'Burgher',
        'Malay': 'Malay',
        'Srilankan Chetty': 'Srilankan Chetty',
        'Bharatha': 'Bharatha',
        'Indian Moor': 'Indian Moor',
        'Pakistan Moor': 'Pakistan Moor',
        'Other Foreigners': 'Other Foreigners',
        'Other Srilankans': 'Other Srilankans'
    }
    
    # Check if it's a numeric code
    if race_str in code_map:
        return code_map[race_str]
    
    # Check if it's a text value
    return text_map.get(race_str, None)

# Create standardized ethnicity columns
df['Mother_Ethnicity_Std'] = df[MOTHER_RACE_COL].apply(standardize_ethnicity)
df['Father_Ethnicity_Std'] = df[FATHER_RACE_COL].apply(standardize_ethnicity)

# Show unique ethnicities found
unique_ethnicities = df['Mother_Ethnicity_Std'].dropna().unique()
print(f"\nUnique ethnicities found in dataset: {sorted(unique_ethnicities)}")

# ============================================
# STEP 3: DISTRICT-LEVEL MISSING BIRTH ORDER ANALYSIS
# ============================================

print("\n📊 STEP 3: District-Level Missing Birth Order Analysis")
print("-" * 80)

# Get unique districts
districts = df[DISTRICT_COL].dropna().unique()
district_summary = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    
    total_births = len(district_data)
    # CHANGED: Now using Missing_Birth_Order
    missing_order = district_data['Missing_Birth_Order'].sum()
    complete_order = total_births - missing_order
    
    district_summary.append({
        'District': district,
        'Total_Births': total_births,
        'Missing_Order': missing_order,
        'Complete_Order': complete_order,
        'Missing_Rate': (missing_order / total_births * 100) if total_births > 0 else 0,
        # NEW: Percentage of total missing records contributed by this district
        'Pct_of_Total_Missing': (missing_order / total_missing * 100) if total_missing > 0 else 0
    })

# Create district summary DataFrame
district_df = pd.DataFrame(district_summary)
district_df = district_df.sort_values('Missing_Rate', ascending=False)

# ============================================
# STEP 4: ETHNICITY DISTRIBUTION BY DISTRICT
# ============================================

print("\n📊 STEP 4: Ethnicity Distribution by District")
print("-" * 80)

# Create detailed ethnicity-district analysis
district_ethnicity_analysis = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    district_total = len(district_data)
    district_missing_total = district_data['Missing_Birth_Order'].sum()
    
    if district_total == 0:
        continue
    
    for ethnicity, config in ETHNICITIES.items():
        ethnic_mothers = district_data[district_data['Mother_Ethnicity_Std'] == ethnicity].shape[0]
        # CHANGED: Now using Missing_Birth_Order
        ethnic_missing = district_data[
            (district_data['Mother_Ethnicity_Std'] == ethnicity) & 
            (district_data['Missing_Birth_Order'] == True)
        ].shape[0]
        
        if ethnic_mothers > 0:
            pct_of_district = (ethnic_mothers / district_total * 100)
            pct_of_district_missing = (ethnic_missing / district_total * 100)
            missing_rate_in_ethnic = (ethnic_missing / ethnic_mothers * 100)
            
            # NEW: Percentage of this district's missing records contributed by this ethnicity
            pct_of_district_missing_records = (ethnic_missing / district_missing_total * 100) if district_missing_total > 0 else 0
            
            # NEW: Percentage of total national missing records contributed by this ethnicity in this district
            pct_of_national_missing = (ethnic_missing / total_missing * 100) if total_missing > 0 else 0
            
            district_ethnicity_analysis.append({
                'District': district,
                'Ethnicity': ethnicity,
                'IUPAC_Code': config['code'],
                'IUPAC_Name': config['iupac_name'],
                'Total_Mothers': ethnic_mothers,
                'Pct_of_District_Total': round(pct_of_district, 2),
                'Missing_Order': ethnic_missing,  # CHANGED
                'Missing_Rate_in_Ethnic': round(missing_rate_in_ethnic, 2),
                'Pct_of_District_Missing': round(pct_of_district_missing, 2),
                # NEW METRICS:
                'Pct_of_District_Missing_Records': round(pct_of_district_missing_records, 2),  # % of this district's missing
                'Pct_of_National_Missing': round(pct_of_national_missing, 2)  # % of total national missing
            })

# Create DataFrame
if district_ethnicity_analysis:
    ethnicity_district_df = pd.DataFrame(district_ethnicity_analysis)
    
    # ============================================
    # CREATE WORD DOCUMENT WITH IUPAC STANDARDS
    # ============================================
    
    print("\n📄 Creating Word Document with IUPAC Standards...")
    
    # Create new document
    doc = Document()
    
    # Set document margins
    sections = doc.sections
    for section in sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)
    
    # Add title - CHANGED
    title = doc.add_heading('Missing Birth Order Analysis Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add subtitle with IUPAC standards
    subtitle = doc.add_heading('Sri Lanka Vital Statistics - IUPAC Compliant Nomenclature', 2)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add date
    date_para = doc.add_paragraph(f'Report Generated: {datetime.now().strftime("%B %d, %Y")}')
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 1: EXECUTIVE SUMMARY
    # ============================================
    
    doc.add_heading('1. Executive Summary', level=1)
    
    # CHANGED: Updated summary text
    summary_text = f"""
    This report presents a comprehensive analysis of missing birth order data in Sri Lanka, 
    stratified by district and ethnicity. The analysis follows IUPAC (International Union of 
    Pure and Applied Chemistry) standards for ethnic nomenclature to ensure international 
    consistency and scientific rigor.
    
    Key Findings:
    • Total Records Analyzed: {total_records:,}
    • Missing Birth Order Records: {total_missing:,} ({total_missing/total_records*100:.2f}%)
    • Complete Records: {total_complete:,} ({total_complete/total_records*100:.2f}%)
    • Number of Districts Analyzed: {len(districts)}
    • Number of Ethnic Groups Identified: {len(unique_ethnicities)}
    
    Note: Birth order refers to the order of birth (first child, second child, etc.) and is a 
    critical demographic variable for understanding fertility patterns and family planning needs.
    """
    
    doc.add_paragraph(summary_text)
    
    # ============================================
    # SECTION 2: IUPAC ETHNICITY CLASSIFICATION
    # ============================================
    
    doc.add_heading('2. IUPAC Ethnicity Classification System', level=1)
    doc.add_paragraph('The following ethnicity codes follow IUPAC standards for population genetics and vital statistics reporting:')
    
    # Create ethnicity table
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Light Grid Accent 1'
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'IUPAC Standard Name'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
        row_cells[2].text = config['iupac_name']
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 3: DISTRICT-WISE ANALYSIS
    # ============================================
    
    doc.add_heading('3. District-Wise Missing Birth Order Analysis', level=1)
    
    # Add district summary table - CHANGED with new metrics
    doc.add_heading('3.1 District Summary Statistics', level=2)
    
    district_table = doc.add_table(rows=1, cols=6)
    district_table.style = 'Light Grid Accent 1'
    hdr_cells = district_table.rows[0].cells
    hdr_cells[0].text = 'District'
    hdr_cells[1].text = 'Total Births'
    hdr_cells[2].text = 'Missing Order Records'
    hdr_cells[3].text = 'Missing Rate (%)'
    hdr_cells[4].text = '% of Total Missing Records'  # NEW
    hdr_cells[5].text = 'Rank by Missing Count'  # NEW
    
    # Sort by missing count for ranking
    district_by_missing = district_df.sort_values('Missing_Order', ascending=False)
    
    for idx, (_, row) in enumerate(district_by_missing.iterrows(), 1):
        row_cells = district_table.add_row().cells
        row_cells[0].text = row['District']
        row_cells[1].text = f"{row['Total_Births']:,}"
        row_cells[2].text = f"{row['Missing_Order']:,}"
        row_cells[3].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[4].text = f"{row['Pct_of_Total_Missing']:.2f}%"
        row_cells[5].text = str(idx)
    
    # Add top districts with highest missing rates
    doc.add_heading('3.2 Districts with Highest Missing Rates', level=2)
    
    top_districts = district_df.head(10)
    top_table = doc.add_table(rows=1, cols=4)
    top_table.style = 'Light Grid Accent 1'
    hdr_cells = top_table.rows[0].cells
    hdr_cells[0].text = 'Rank by Rate'
    hdr_cells[1].text = 'District'
    hdr_cells[2].text = 'Missing Rate (%)'
    hdr_cells[3].text = '% of Total Missing'
    
    for idx, (_, row) in enumerate(top_districts.iterrows(), 1):
        row_cells = top_table.add_row().cells
        row_cells[0].text = str(idx)
        row_cells[1].text = row['District']
        row_cells[2].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[3].text = f"{row['Pct_of_Total_Missing']:.2f}%"
    
    # NEW SECTION: Districts with highest missing counts (not just rates)
    doc.add_heading('3.3 Districts with Highest Missing Counts', level=2)
    
    top_districts_by_count = district_df.nlargest(10, 'Missing_Order')
    count_table = doc.add_table(rows=1, cols=4)
    count_table.style = 'Light Grid Accent 1'
    hdr_cells = count_table.rows[0].cells
    hdr_cells[0].text = 'Rank by Count'
    hdr_cells[1].text = 'District'
    hdr_cells[2].text = 'Missing Order Count'
    hdr_cells[3].text = 'Missing Rate (%)'
    
    for idx, (_, row) in enumerate(top_districts_by_count.iterrows(), 1):
        row_cells = count_table.add_row().cells
        row_cells[0].text = str(idx)
        row_cells[1].text = row['District']
        row_cells[2].text = f"{row['Missing_Order']:,}"
        row_cells[3].text = f"{row['Missing_Rate']:.2f}%"
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 4: DETAILED DISTRICT-ETHNICITY ANALYSIS
    # ============================================
    
    doc.add_heading('4. Detailed District-Ethnicity Analysis', level=1)
    doc.add_paragraph('The following tables show the ethnicity distribution and missing birth order patterns for each district, following IUPAC nomenclature.')
    
    for district in districts[:15]:  # Show first 15 districts (adjust as needed)
        district_ethnic = ethnicity_district_df[ethnicity_district_df['District'] == district]
        
        if len(district_ethnic) > 0:
            district_total = district_df[district_df['District'] == district]['Total_Births'].values[0]
            district_missing = district_df[district_df['District'] == district]['Missing_Order'].values[0]
            district_pct_missing = district_df[district_df['District'] == district]['Pct_of_Total_Missing'].values[0]
            
            # Add district header
            doc.add_heading(f'{district}', level=2)
            doc.add_paragraph(f'Total Births: {district_total:,} | Missing Order: {district_missing:,} ({district_missing/district_total*100:.2f}%) | {district_pct_missing:.1f}% of National Missing Records')
            
            # Create ethnicity table for district - CHANGED with new columns
            ethnic_table = doc.add_table(rows=1, cols=9)
            ethnic_table.style = 'Light Grid Accent 1'
            hdr_cells = ethnic_table.rows[0].cells
            hdr_cells[0].text = 'IUPAC Code'
            hdr_cells[1].text = 'Ethnicity'
            hdr_cells[2].text = 'Count'
            hdr_cells[3].text = '% of District'
            hdr_cells[4].text = 'Missing Order'
            hdr_cells[5].text = 'Missing Rate (%)'
            hdr_cells[6].text = '% of District Missing Records'  # NEW
            hdr_cells[7].text = '% of National Missing'  # NEW
            hdr_cells[8].text = 'Rank in District'  # NEW
            
            district_ethnic_sorted = district_ethnic.sort_values('Pct_of_District_Total', ascending=False)
            
            # Add rank by missing count within district
            district_ethnic_sorted = district_ethnic_sorted.copy()
            district_ethnic_sorted['Rank_In_District'] = district_ethnic_sorted['Missing_Order'].rank(ascending=False, method='dense').astype(int)
            
            for _, row in district_ethnic_sorted.iterrows():
                row_cells = ethnic_table.add_row().cells
                row_cells[0].text = row['IUPAC_Code']
                row_cells[1].text = row['Ethnicity']
                row_cells[2].text = f"{row['Total_Mothers']:,}"
                row_cells[3].text = f"{row['Pct_of_District_Total']:.2f}%"
                row_cells[4].text = f"{row['Missing_Order']:,}"  # CHANGED
                row_cells[5].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[6].text = f"{row['Pct_of_District_Missing_Records']:.2f}%"  # NEW
                row_cells[7].text = f"{row['Pct_of_National_Missing']:.2f}%"  # NEW
                row_cells[8].text = str(row['Rank_In_District'])  # NEW
            
            # Add district summary
            doc.add_paragraph()
            most_prevalent = district_ethnic_sorted.iloc[0]
            highest_missing = district_ethnic_sorted.loc[district_ethnic_sorted['Missing_Rate_in_Ethnic'].idxmax()]
            largest_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_District_Missing_Records'].idxmax()]
            largest_national_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_National_Missing'].idxmax()]
            
            summary_para = doc.add_paragraph()
            summary_para.add_run('District Summary:').bold = True
            doc.add_paragraph(f'• Most prevalent ethnicity: {most_prevalent["Ethnicity"]} ({most_prevalent["IUPAC_Code"]}) - {most_prevalent["Pct_of_District_Total"]:.1f}% of district')
            doc.add_paragraph(f'• Highest missing rate: {highest_missing["Ethnicity"]} ({highest_missing["IUPAC_Code"]}) - {highest_missing["Missing_Rate_in_Ethnic"]:.1f}% missing')
            doc.add_paragraph(f'• Largest contributor to district missing data: {largest_contributor["Ethnicity"]} ({largest_contributor["IUPAC_Code"]}) - {largest_contributor["Pct_of_District_Missing_Records"]:.1f}% of district missing records')
            doc.add_paragraph(f'• Largest contributor to national missing data: {largest_national_contributor["Ethnicity"]} ({largest_national_contributor["IUPAC_Code"]}) - {largest_national_contributor["Pct_of_National_Missing"]:.1f}% of national missing records')
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 5: ETHNICITY-SPECIFIC ANALYSIS
    # ============================================
    
    doc.add_heading('5. Ethnicity-Specific Analysis', level=1)
    
    # Calculate overall ethnicity statistics - CHANGED with new metrics
    ethnicity_overall = ethnicity_district_df.groupby(['Ethnicity', 'IUPAC_Code', 'IUPAC_Name']).agg({
        'Total_Mothers': 'sum',
        'Missing_Order': 'sum',  # CHANGED
        'Pct_of_National_Missing': 'sum'  # NEW: Sum of national percentages
    }).reset_index()
    ethnicity_overall['Missing_Rate'] = (ethnicity_overall['Missing_Order'] / ethnicity_overall['Total_Mothers'] * 100)
    ethnicity_overall = ethnicity_overall.sort_values('Missing_Rate', ascending=False)
    
    doc.add_heading('5.1 Overall Ethnicity Statistics', level=2)
    
    overall_table = doc.add_table(rows=1, cols=6)
    overall_table.style = 'Light Grid Accent 1'
    hdr_cells = overall_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'Total Births'
    hdr_cells[3].text = 'Missing Order Records'
    hdr_cells[4].text = 'Missing Rate (%)'
    hdr_cells[5].text = '% of National Missing'  # NEW
    
    for _, row in ethnicity_overall.iterrows():
        row_cells = overall_table.add_row().cells
        row_cells[0].text = row['IUPAC_Code']
        row_cells[1].text = row['Ethnicity']
        row_cells[2].text = f"{row['Total_Mothers']:,}"
        row_cells[3].text = f"{row['Missing_Order']:,}"  # CHANGED
        row_cells[4].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[5].text = f"{row['Pct_of_National_Missing']:.2f}%"  # NEW
    
    # Add ethnicity-specific district analysis
    doc.add_heading('5.2 Ethnicity-Specific District Analysis', level=2)
    
    for ethnicity in list(ETHNICITIES.keys())[:6]:  # Show first 6 ethnicities
        ethnic_data = ethnicity_district_df[ethnicity_district_df['Ethnicity'] == ethnicity]
        
        if len(ethnic_data) > 0:
            ethnic_total = ethnic_data['Total_Mothers'].sum()
            ethnic_missing = ethnic_data['Missing_Order'].sum()
            ethnic_rate = (ethnic_missing / ethnic_total * 100) if ethnic_total > 0 else 0
            ethnic_national_pct = ethnic_data['Pct_of_National_Missing'].sum()
            
            doc.add_heading(f'{ETHNICITIES[ethnicity]["full_name"]} ({ETHNICITIES[ethnicity]["code"]})', level=3)
            doc.add_paragraph(f'Overall Statistics: Total Births: {ethnic_total:,} | Missing Order: {ethnic_missing:,} ({ethnic_rate:.2f}%) | {ethnic_national_pct:.1f}% of National Missing Records')
            
            # Create table for districts with highest missing rates AND highest missing counts for this ethnicity
            doc.add_paragraph('Districts with Highest Missing Rates for this Ethnicity:', style='List Bullet')
            rate_table = doc.add_table(rows=1, cols=5)
            rate_table.style = 'Light Grid Accent 1'
            hdr_cells = rate_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Order'
            hdr_cells[3].text = 'Missing Rate (%)'
            hdr_cells[4].text = '% of Ethnic Missing'
            
            ethnic_sorted_by_rate = ethnic_data.sort_values('Missing_Rate_in_Ethnic', ascending=False).head(10)
            
            for _, row in ethnic_sorted_by_rate.iterrows():
                row_cells = rate_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Order']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[4].text = f"{(row['Missing_Order']/ethnic_missing*100):.2f}%"
            
            doc.add_paragraph()
            
            # NEW: Districts with highest missing counts for this ethnicity
            doc.add_paragraph('Districts with Highest Missing Counts for this Ethnicity:', style='List Bullet')
            count_table = doc.add_table(rows=1, cols=5)
            count_table.style = 'Light Grid Accent 1'
            hdr_cells = count_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Order'
            hdr_cells[3].text = 'Missing Rate (%)'
            hdr_cells[4].text = '% of Ethnic Missing'
            
            ethnic_sorted_by_count = ethnic_data.sort_values('Missing_Order', ascending=False).head(10)
            
            for _, row in ethnic_sorted_by_count.iterrows():
                row_cells = count_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Order']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[4].text = f"{(row['Missing_Order']/ethnic_missing*100):.2f}%"
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 6: CONCLUSIONS AND RECOMMENDATIONS
    # ============================================
    
    doc.add_heading('6. Conclusions and Recommendations', level=1)
    
    # Find key insights
    worst_district_by_rate = district_df.iloc[0]
    worst_district_by_count = district_df.nlargest(1, 'Missing_Order').iloc[0]
    worst_ethnicity = ethnicity_overall.iloc[0]
    
    # Find which ethnicity contributes most to national missing
    top_national_contributor = ethnicity_overall.nlargest(1, 'Pct_of_National_Missing').iloc[0]
    
    # CHANGED: Updated conclusions
    conclusions = f"""
    6.1 Key Findings
    
    • Data Completeness: {total_complete/total_records*100:.2f}% of birth records have complete birth order data,
      indicating {total_missing/total_records*100:.2f}% of records require data quality improvement.
    
    • Geographic Disparities by Rate: {worst_district_by_rate['District']} shows the highest missing rate for birth order at 
      {worst_district_by_rate['Missing_Rate']:.2f}%, suggesting potential data collection challenges in this region.
    
    • Geographic Disparities by Count: {worst_district_by_count['District']} has the highest absolute number of missing records
      ({worst_district_by_count['Missing_Order']:,} records, representing {worst_district_by_count['Pct_of_Total_Missing']:.1f}% of all missing data).
    
    • Ethnic Disparities by Rate: {worst_ethnicity['Ethnicity']} ({worst_ethnicity['IUPAC_Code']}) has the highest 
      missing rate for birth order at {worst_ethnicity['Missing_Rate']:.2f}%, indicating potential systematic bias 
      in data collection across ethnic groups.
    
    • Ethnic Disparities by Contribution: {top_national_contributor['Ethnicity']} ({top_national_contributor['IUPAC_Code']}) 
      contributes the largest share ({top_national_contributor['Pct_of_National_Missing']:.1f}%) of all missing birth order records nationally.
    
    • Public Health Implications: Missing birth order data affects the accuracy of fertility analyses, 
      family planning assessments, and maternal and child health program evaluations.
    
    6.2 Recommendations
    
    1. Prioritize High-Volume Districts: Focus data quality improvement efforts first on districts with the highest
       absolute missing counts ({worst_district_by_count['District']}, {worst_district_by_count['Missing_Order']:,} missing records).
    
    2. Target High-Rate Districts: Implement specialized interventions in districts with the highest missing rates
       ({worst_district_by_rate['District']}: {worst_district_by_rate['Missing_Rate']:.1f}% missing).
    
    3. Ethnicity-Specific Interventions: Develop targeted data quality improvement programs for ethnic groups 
       with both high missing rates ({worst_ethnicity['Ethnicity']}: {worst_ethnicity['Missing_Rate']:.1f}%) 
       and high contribution to national missing ({top_national_contributor['Ethnicity']}: {top_national_contributor['Pct_of_National_Missing']:.1f}%).
    
    4. IUPAC Compliance: Maintain adherence to IUPAC ethnic nomenclature standards to ensure international 
       comparability and scientific rigor.
    
    5. Regular Monitoring: Establish quarterly data quality monitoring systems to track improvements in 
       missing birth order rates by district and ethnicity, focusing on both rates and absolute counts.
    
    6. Capacity Building: Provide training for healthcare workers on the importance of complete birth order 
       documentation, especially in high-volume and high-rate districts.
    
    7. Electronic Health Records: Implement or strengthen electronic health record systems with mandatory 
       birth order fields to reduce missing data.
    """
    
    doc.add_paragraph(conclusions)
    
    # Add methodology section
    doc.add_heading('7. Methodology', level=1)
    
    methodology = f"""
    This analysis was conducted using vital statistics data from Sri Lanka. The methodology follows 
    IUPAC standards for ethnic classification and WHO guidelines for demographic data collection.
    
    Data Sources:
    • Birth Registration Records: {total_records:,} records
    • Time Period: Full dataset analysis
    • Geographic Coverage: {len(districts)} districts
    
    Key Metrics Defined:
    
    1. Missing Rate: Percentage of records missing birth order within a specific group
       Formula: (Missing Order / Total Births) × 100
    
    2. Percentage of Total Missing Records: What proportion of ALL national missing records 
       comes from a specific district or ethnicity
       Formula: (Missing Order in Group / Total National Missing) × 100
    
    3. Percentage of District Missing Records: What proportion of a district's missing records 
       comes from a specific ethnicity
       Formula: (Missing Order for Ethnicity in District / Total District Missing) × 100
    
    4. Percentage of National Missing: What proportion of ALL national missing records comes 
       from a specific ethnicity-district combination
    
    Birth Order Definition:
    Birth order refers to the numerical order of a child's birth among all live births to the same mother 
    (first child, second child, third child, etc.). Complete birth order data is essential for:
    • Fertility rate calculations
    • Family planning program evaluation
    • Maternal health risk assessment
    • Demographic transition analysis
    
    Ethnic Classification:
    Ethnicities were classified according to IUPAC standards using the following codes:
    """
    
    doc.add_paragraph(methodology)
    
    # Add IUPAC code reference
    ref_table = doc.add_table(rows=1, cols=2)
    ref_table.style = 'Light Grid Accent 1'
    hdr_cells = ref_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnic Group'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = ref_table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
    
    # Save the document - CHANGED filename
    filename = f'Missing_Birth_Order_Analysis_IUPAC_{datetime.now().strftime("%Y%m%d_%H%M%S")}.docx'
    doc.save(filename)
    print(f"\n✅ Word document saved as: {filename}")
    
    # ============================================
    # STEP 5: DISPLAY SUMMARY IN CONSOLE
    # ============================================
    
    print("\n" + "=" * 100)
    print("📊 FINAL SUMMARY: Missing Birth Order Analysis by District and Ethnicity")
    print("=" * 100)
    
    print(f"\n✅ Analysis Complete!")
    print(f"✅ Word Document Generated: {filename}")
    print(f"\nOverall Statistics:")
    print(f"   • Total Districts: {len(districts)}")
    print(f"   • Total Births: {total_records:,}")
    print(f"   • Total Missing Birth Order: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    
    print(f"\n🏆 Top 5 Districts with Highest Missing RATE:")
    for _, row in district_df.head(5).iterrows():
        print(f"   • {row['District']}: {row['Missing_Rate']:.2f}% ({row['Missing_Order']:,}/{row['Total_Births']:,})")
    
    print(f"\n🏆 Top 5 Districts with Highest Missing COUNT (absolute):")
    district_by_count = district_df.nlargest(5, 'Missing_Order')
    for _, row in district_by_count.iterrows():
        print(f"   • {row['District']}: {row['Missing_Order']:,} missing records ({row['Missing_Rate']:.2f}% of district)")
    
    print(f"\n🏆 Top 5 Districts by % of Total National Missing:")
    district_by_pct = district_df.nlargest(5, 'Pct_of_Total_Missing')
    for _, row in district_by_pct.iterrows():
        print(f"   • {row['District']}: {row['Pct_of_Total_Missing']:.2f}% of national missing ({row['Missing_Order']:,} records)")
    
    print(f"\n🏆 Top 5 Ethnicities with Highest Missing RATE (Overall):")
    for _, row in ethnicity_overall.head(5).iterrows():
        print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Missing_Rate']:.2f}% ({row['Missing_Order']:,}/{row['Total_Mothers']:,})")
    
    print(f"\n🏆 Top 5 Ethnicities by % of Total National Missing:")
    ethnicity_by_pct = ethnicity_overall.nlargest(5, 'Pct_of_National_Missing')
    for _, row in ethnicity_by_pct.iterrows():
        print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Pct_of_National_Missing']:.2f}% of national missing ({row['Missing_Order']:,} records)")

else:
    print("\n⚠️ No ethnicity data found. Please check the ethnicity column values.")
    print(f"Sample values from {MOTHER_RACE_COL}:")
    print(df[MOTHER_RACE_COL].value_counts().head(10))

print("\n✅ Analysis Complete!")

🏥 MISSING BIRTH ORDER ANALYSIS BY DISTRICT AND ETHNICITY

📊 STEP 1: Missing Birth Order Analysis
--------------------------------------------------------------------------------

Overall Statistics:
   • Total records: 301,691
   • Missing birth order: 27,213 (9.02%)
   • Complete birth order: 274,478 (90.98%)

📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)
--------------------------------------------------------------------------------

Unique ethnicities found in dataset: ['Burgher', 'Indian Tamil', 'Malay', 'Sinhalese', 'Srilankan Moor', 'Srilankan Tamil']

📊 STEP 3: District-Level Missing Birth Order Analysis
--------------------------------------------------------------------------------

📊 STEP 4: Ethnicity Distribution by District
--------------------------------------------------------------------------------

📄 Creating Word Document with IUPAC Standards...

✅ Word document saved as: Missing_Birth_Order_Analysis_IUPAC_20260501_201406.docx

📊 FINAL SUMMARY: Missing Birth

In [6]:
# ============================================
# SECTION 4: DETAILED DISTRICT-ETHNICITY ANALYSIS (TOP 15 BY MISSING RATE)
# ============================================

doc.add_heading('4. Detailed District-Ethnicity Analysis', level=1)
doc.add_paragraph('The following tables show the ethnicity distribution and missing birth order patterns for the top 15 districts with the highest missing rates, following IUPAC nomenclature.')

# Get top 15 districts by missing rate
top_districts_by_rate = district_df.nlargest(15, 'Missing_Rate')

for district in top_districts_by_rate['District'].values:
    district_ethnic = ethnicity_district_df[ethnicity_district_df['District'] == district]
    
    if len(district_ethnic) > 0:
        district_total = district_df[district_df['District'] == district]['Total_Births'].values[0]
        district_missing = district_df[district_df['District'] == district]['Missing_Order'].values[0]
        district_pct_missing = district_df[district_df['District'] == district]['Pct_of_Total_Missing'].values[0]
        
        # Add district header
        doc.add_heading(f'{district}', level=2)
        doc.add_paragraph(f'Total Births: {district_total:,} | Missing Order: {district_missing:,} ({district_missing/district_total*100:.2f}%) | {district_pct_missing:.1f}% of National Missing Records')
        
        # Create ethnicity table for district - CHANGED with new columns
        ethnic_table = doc.add_table(rows=1, cols=9)
        ethnic_table.style = 'Light Grid Accent 1'
        hdr_cells = ethnic_table.rows[0].cells
        hdr_cells[0].text = 'IUPAC Code'
        hdr_cells[1].text = 'Ethnicity'
        hdr_cells[2].text = 'Count'
        hdr_cells[3].text = '% of District'
        hdr_cells[4].text = 'Missing Order'
        hdr_cells[5].text = 'Missing Rate (%)'
        hdr_cells[6].text = '% of District Missing Records'  # NEW
        hdr_cells[7].text = '% of National Missing'  # NEW
        hdr_cells[8].text = 'Rank in District'  # NEW
        
        district_ethnic_sorted = district_ethnic.sort_values('Pct_of_District_Total', ascending=False)
        
        # Add rank by missing count within district
        district_ethnic_sorted = district_ethnic_sorted.copy()
        district_ethnic_sorted['Rank_In_District'] = district_ethnic_sorted['Missing_Order'].rank(ascending=False, method='dense').astype(int)
        
        for _, row in district_ethnic_sorted.iterrows():
            row_cells = ethnic_table.add_row().cells
            row_cells[0].text = row['IUPAC_Code']
            row_cells[1].text = row['Ethnicity']
            row_cells[2].text = f"{row['Total_Mothers']:,}"
            row_cells[3].text = f"{row['Pct_of_District_Total']:.2f}%"
            row_cells[4].text = f"{row['Missing_Order']:,}"  # CHANGED
            row_cells[5].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
            row_cells[6].text = f"{row['Pct_of_District_Missing_Records']:.2f}%"  # NEW
            row_cells[7].text = f"{row['Pct_of_National_Missing']:.2f}%"  # NEW
            row_cells[8].text = str(row['Rank_In_District'])  # NEW
        
        # Add district summary
        doc.add_paragraph()
        most_prevalent = district_ethnic_sorted.iloc[0]
        highest_missing = district_ethnic_sorted.loc[district_ethnic_sorted['Missing_Rate_in_Ethnic'].idxmax()]
        largest_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_District_Missing_Records'].idxmax()]
        largest_national_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_National_Missing'].idxmax()]
        
        summary_para = doc.add_paragraph()
        summary_para.add_run('District Summary:').bold = True
        doc.add_paragraph(f'• Most prevalent ethnicity: {most_prevalent["Ethnicity"]} ({most_prevalent["IUPAC_Code"]}) - {most_prevalent["Pct_of_District_Total"]:.1f}% of district')
        doc.add_paragraph(f'• Highest missing rate: {highest_missing["Ethnicity"]} ({highest_missing["IUPAC_Code"]}) - {highest_missing["Missing_Rate_in_Ethnic"]:.1f}% missing')
        doc.add_paragraph(f'• Largest contributor to district missing data: {largest_contributor["Ethnicity"]} ({largest_contributor["IUPAC_Code"]}) - {largest_contributor["Pct_of_District_Missing_Records"]:.1f}% of district missing records')
        doc.add_paragraph(f'• Largest contributor to national missing data: {largest_national_contributor["Ethnicity"]} ({largest_national_contributor["IUPAC_Code"]}) - {largest_national_contributor["Pct_of_National_Missing"]:.1f}% of national missing records')
        
        doc.add_paragraph()